In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(override=True)

True

In [3]:
doc_loader = TextLoader("../data/text_files/python_intro.txt")

raw_docs = doc_loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(raw_docs)

embedding = HuggingFaceEmbeddings(model_name = os.getenv("HUGGING_FACE_EMBEDDING_MODEL"))
vector_store = FAISS.from_documents(chunks, embedding)

retriever = vector_store.as_retriever(search_type = "mmr", search_kwargs = {"k":4, "lambda_mult":0.7})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12951.20it/s]


In [4]:
llm = init_chat_model(model=os.getenv("OLLAMA_CHAT_MODEL"))
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [5]:
decomposition_prompt = PromptTemplate.from_template("""
You are as AI assistant. Decompose the following compled question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [6]:
query = "How does python help build AI and ML applications?"

decomposed_query = decomposition_chain.invoke({"question": query})

print(decomposed_query)

**Sub-questions:**

1. **What key Python libraries are used in AI and ML development (e.g., NumPy, Pandas, Scikit-learn)?**
   *(Focuses retrieval on the essential tools, frameworks, and foundational packages.)*

2. **How does Python enable data preprocessing and exploratory data analysis (EDA) for ML models?**
   *(Focuses retrieval on the critical initial steps—handling data, which is often the hardest part of an ML project.)*

3. **How is Python used for deep learning and building complex neural networks (e.g., using TensorFlow or PyTorch)?**
   *(Focuses retrieval on the most advanced aspect of AI, machine learning, and specialized deep learning frameworks.)*

4. **What is the typical end-to-end workflow for building an ML application using Python?**
   *(Provides a high-level roadmap, connecting the concepts from data handling to model deployment.)*


In [7]:
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")

qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [8]:
def full_query_decomposition_rag_pipeline(user_query):
    sub_qa_text = decomposition_chain.invoke({"question": user_query})

    sub_questions = [q.strip("-.123467890. ").strip() for q in sub_qa_text.split("\n") if q.strip()]

    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")

    return "\n\n".join(results)

In [9]:
query = "How does python help build AI and ML applications?"

final_answer = full_query_decomposition_rag_pipeline(query)

print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: **Sub-questions:**
A: Please provide the specific question you would like me to answer.

The text snippet contains several distinct pieces of information (criticism of Python, the Zen of Python, the `raise` statement), but there is no question provided to base an answer on.

Q: **What Python libraries are essential for data preprocessing and manipulation in ML projects?** (Focuses on the foundational step: data handling. *Retrieval target: Pandas, NumPy.*)
A: Based on the provided context, the names of the libraries Pandas and NumPy are not mentioned, so I cannot answer the question.

Q: **How does Python facilitate the development of deep learning models (e.g., using TensorFlow or PyTorch)?** (Focuses on the most complex and powerful part of AI. *Retrieval target: Specific frameworks, architecture implementation.*)
A: Based on the provided context, there is no information available regarding how Python facilitates the development of deep learning models using speci